# Test Sentence Encoders on Apple Silicon

This notebook tests the sentence-transformers based encoders that are compatible with Apple Silicon (M1/M2/M3).

In [ ]:
import os
import platform
import time

import numpy as np

# Check platform
print(f"Platform: {platform.system()}")
print(f"Machine: {platform.machine()}")
print(
    f"Apple Silicon: {platform.system() == 'Darwin' and platform.machine() == 'arm64'}"
)

## 1. Test DistilUSE Encoder

Knowledge-distilled version of Universal Sentence Encoder. 512-dimensional embeddings.

In [ ]:
from aymurai.models.usem import DistilUSEEncoder

distiluse = DistilUSEEncoder()
print(f"Model: {distiluse.MODEL_NAME}")
print(f"Device: {distiluse.model.device}")

In [ ]:
# Test encoding
test_sentences = [
    "El juez dictó sentencia en el caso de violencia de género.",
    "La víctima presentó una denuncia por amenazas.",
    "Se otorgó una medida de protección a la denunciante.",
    "The judge issued a ruling in the gender violence case.",
    "The victim filed a complaint for threats.",
]

start = time.time()
embeddings_distiluse = distiluse.encode(test_sentences, encoder_type="question_encoder")
elapsed = time.time() - start

print(f"Embeddings shape: {embeddings_distiluse.shape}")
print(f"Encoding time: {elapsed:.3f}s")

## 2. Test Multilingual MiniLM Encoder

Higher quality embeddings, faster inference. 384-dimensional embeddings.

In [ ]:
from aymurai.models.usem import MultilingualMiniLMEncoder

minilm = MultilingualMiniLMEncoder()
print(f"Model: {minilm.MODEL_NAME}")
print(f"Device: {minilm.model.device}")

In [ ]:
start = time.time()
embeddings_minilm = minilm.encode(test_sentences, encoder_type="question_encoder")
elapsed = time.time() - start

print(f"Embeddings shape: {embeddings_minilm.shape}")
print(f"Encoding time: {elapsed:.3f}s")

## 3. Test Factory Auto-Detection

In [ ]:
from aymurai.models.usem import create_encoder, EncoderType

# Auto-detect (should use DistilUSE on Apple Silicon)
encoder_auto = create_encoder()
print(f"Auto-detected encoder: {type(encoder_auto).__name__}")

# Explicit selection
encoder_minilm = create_encoder(EncoderType.MINILM)
print(f"Explicit MiniLM encoder: {type(encoder_minilm).__name__}")

## 4. Semantic Similarity Test

Compare similarity between Spanish and English sentences.

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def similarity_matrix(embeddings, sentences):
    """Compute and display similarity matrix."""
    n = len(sentences)
    sim_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim_matrix[i, j] = cosine_similarity(embeddings[i], embeddings[j])
    return sim_matrix

In [ ]:
# Compute similarity matrix for DistilUSE
sim_distiluse = similarity_matrix(embeddings_distiluse, test_sentences)

print("DistilUSE Similarity Matrix:")
print("Sentences:")
for i, s in enumerate(test_sentences):
    print(f"  [{i}] {s[:60]}..." if len(s) > 60 else f"  [{i}] {s}")
print()
print(np.round(sim_distiluse, 3))

In [ ]:
# Compute similarity matrix for MiniLM
sim_minilm = similarity_matrix(embeddings_minilm, test_sentences)

print("MiniLM Similarity Matrix:")
print(np.round(sim_minilm, 3))

## 5. Cross-lingual Similarity

Test that Spanish-English translation pairs have high similarity.

In [ ]:
# Spanish sentence [0] should be similar to English sentence [3]
# Spanish sentence [1] should be similar to English sentence [4]

print("Cross-lingual similarity (Spanish ↔ English):")
print(f"\nDistilUSE:")
print(
    f"  '{test_sentences[0][:40]}...' ↔ '{test_sentences[3][:40]}...': {sim_distiluse[0, 3]:.3f}"
)
print(
    f"  '{test_sentences[1][:40]}...' ↔ '{test_sentences[4][:40]}...': {sim_distiluse[1, 4]:.3f}"
)

print(f"\nMiniLM:")
print(
    f"  '{test_sentences[0][:40]}...' ↔ '{test_sentences[3][:40]}...': {sim_minilm[0, 3]:.3f}"
)
print(
    f"  '{test_sentences[1][:40]}...' ↔ '{test_sentences[4][:40]}...': {sim_minilm[1, 4]:.3f}"
)

## 6. Batch Encoding Performance

In [ ]:
# Generate more sentences for batch test
batch_sentences = test_sentences * 100  # 500 sentences

print(f"Batch size: {len(batch_sentences)} sentences")

# DistilUSE batch
start = time.time()
batch_embeddings_distiluse = distiluse.batch_encode(
    batch_sentences, encoder_type="question_encoder", batch_size=64
)
elapsed_distiluse = time.time() - start
print(
    f"\nDistilUSE batch encoding: {elapsed_distiluse:.3f}s ({len(batch_sentences) / elapsed_distiluse:.1f} sent/s)"
)

# MiniLM batch
start = time.time()
batch_embeddings_minilm = minilm.batch_encode(
    batch_sentences, encoder_type="question_encoder", batch_size=64
)
elapsed_minilm = time.time() - start
print(
    f"MiniLM batch encoding: {elapsed_minilm:.3f}s ({len(batch_sentences) / elapsed_minilm:.1f} sent/s)"
)

## 7. Environment Variable Configuration

In [ ]:
# Test environment variable configuration
import importlib
import aymurai.models.usem.factory as factory_module

# Set env var and reload
os.environ["SENTENCE_ENCODER_TYPE"] = "minilm"
importlib.reload(factory_module)
from aymurai.models.usem.factory import create_encoder

encoder_from_env = create_encoder()
print(f"Encoder from SENTENCE_ENCODER_TYPE=minilm: {type(encoder_from_env).__name__}")

# Reset to auto
os.environ["SENTENCE_ENCODER_TYPE"] = "auto"

## Summary

Both `DistilUSEEncoder` and `MultilingualMiniLMEncoder` work on Apple Silicon:

| Model | Embedding Dim | Speed | Quality |
|-------|---------------|-------|--------|
| DistilUSE | 512 | Good | Good (USE distillation) |
| MiniLM | 384 | Faster | Better |

Use `SENTENCE_ENCODER_TYPE=auto` for automatic platform detection.